# Jobs Class Analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

path = '/content/drive/My Drive/ghdp_7031_data.csv'
df = pd.read_csv(path)

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

In [ ]:
# Summary table, 1 decimal
summary = df.describe().T
print(summary.to_string(float_format='{:,.1f}'.format))

In [ ]:
# Poverty rate (%) by YEAR and work status last year, weighted with ASECWT
work = df['FULLTIME_LABEL'].fillna('Did not work last year')
poor = (df['POOR'] == 1).astype(float)

weighted_poor = (poor * df['ASECWT']).groupby([df['YEAR'], work]).sum()
weighted_total = df['ASECWT'].groupby([df['YEAR'], work]).sum()
poverty_rate = (weighted_poor / weighted_total * 100).unstack()
poverty_rate.index.name = 'Year'
poverty_rate.columns.name = 'Work status last year'

print('Poverty rate (% of people who are poor), weighted with ASECWT')
print(poverty_rate.to_string(float_format='{:.1f}'.format))

# Unweighted number of respondents in each cell
counts = pd.crosstab(df['YEAR'], work)
counts.index.name = 'Year'
counts.columns.name = 'Work status last year'
print('\nUnweighted number of respondents in each cell')
print(counts.to_string())

# What one cell means
year, status = poverty_rate.index[0], poverty_rate.columns[0]
print(f'\nHow to read one cell: in {year}, among people whose work status last year was '
      f'"{status}", an estimated {poverty_rate.loc[year, status]:.1f}% were poor. '
      f'The weights (ASECWT) make this an estimate for the whole population, '
      f'not just the {counts.loc[year, status]:,} people surveyed in that cell.')